## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T), lo que evita desbordes numéricos sin alterar el resultado. Con los logits z = [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] para los tokens t0…t5, las distribuciones para T = 0.5, T = 1 y T = 2, junto con sus entropías en bits, son:

| T | t0 | t1 | t2 | t3 | t4 | t5 | entropía (bits) | suma probs |
|---|---|---|---|---|---|---|---|---|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 | 1.0000 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 | 1.0000 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 | 1.0000 |

Las tres filas suman 1.0000, lo que verifica la normalización. Al aumentar T la masa de probabilidad se reparte desde t0 hacia el resto del vocabulario: la probabilidad de t0 cae de 0.8231 (T = 0.5) a 0.5516 (T = 1) y a 0.3583 (T = 2), mientras que la de t5 sube de 3.7367e-05 (que se muestra como 0.0000 a cuatro decimales) a 0.0037 y luego a 0.0294. En consecuencia, la entropía crece monótonamente con la temperatura: 0.9147 → 1.7998 → 2.2888 bits. La figura generada en la celda muestra las tres distribuciones lado a lado.

In [1]:
# T1: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura (numéricamente estable) y entropía en bits.

Implementa en NumPy p_i = exp(z_i/T) / Σ_j exp(z_j/T) restando el máximo de los
logits escalados (estabilidad numérica), calcula la distribución sobre los seis
tokens para T = 0.5, 1 y 2, la entropía en bits de cada distribución, y presenta
todo en una tabla con cuatro decimales. Guarda los resultados en resultados.json
y una figura PNG con las tres distribuciones.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# 1. Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------
# 2. Softmax con temperatura, numéricamente estable
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    """
    Softmax con temperatura p_i = exp(z_i/T) / Σ_j exp(z_j/T).

    Estabilidad numérica: se resta el máximo de los logits escalados (z/T)
    antes de la exponencial, de modo que el mayor exponente es 0 y no hay
    desbordamiento (overflow) en exp.
    """
    z = np.asarray(z, dtype=float)
    x = z / float(T)
    x = x - np.max(x)          # truco de estabilidad: max(x) se vuelve 0
    e = np.exp(x)
    return e / np.sum(e)


def entropia_bits(p):
    """Entropía de Shannon H = -Σ p_i log2(p_i), en bits."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]               # evita log2(0); 0*log2(0) = 0 por convención
    return float(-np.sum(p * np.log2(p)))


# ----------------------------------------------------------------------
# 3. Cálculo de las tres distribuciones y sus entropías
# ----------------------------------------------------------------------
distribuciones = {}
entropias = {}
sumas = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))

# ----------------------------------------------------------------------
# 4. Tabla con cuatro decimales (generada por el código, no a mano)
# ----------------------------------------------------------------------
filas = []
for T in temperaturas:
    filas.append(
        [f"{T:.1f}"]
        + [f"{v:.4f}" for v in distribuciones[T]]
        + [f"{entropias[T]:.4f}", f"{sumas[T]:.4f}"]
    )
tabla_df = pd.DataFrame(
    filas, columns=["T"] + tokens + ["entropia_bits", "suma_probs"]
)

# Versión markdown de la tabla (para resultados.json / notebook)
encabezado = "| " + " | ".join(tabla_df.columns) + " |"
separador = "|" + "|".join(["---"] * len(tabla_df.columns)) + "|"
cuerpo = "\n".join("| " + " | ".join(fila) + " |" for fila in filas)
tabla_markdown = "\n".join([encabezado, separador, cuerpo])

print("Softmax con temperatura (distribuciones y entropías, 4 decimales):\n")
print(tabla_df.to_string(index=False))
print("\nTabla markdown:\n")
print(tabla_markdown)

# ----------------------------------------------------------------------
# 5. Figura PNG: las tres distribuciones
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
ancho = 0.25
posiciones = np.arange(len(tokens))
for i, T in enumerate(temperaturas):
    ax.bar(
        posiciones + (i - 1) * ancho,
        distribuciones[T],
        width=ancho,
        label=f"T = {T:g}",
    )
ax.set_xticks(posiciones)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura — logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("t1_distribuciones_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 6. resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
claves_T = {0.5: "T=0.5", 1.0: "T=1", 2.0: "T=2"}
resultados = {
    "subtarea": "T1_softmax_temperatura",
    "tokens": tokens,
    "logits": [float(v) for v in logits],
    "temperaturas": [float(T) for T in temperaturas],
    "distribuciones": {
        claves_T[T]: [float(v) for v in distribuciones[T]] for T in temperaturas
    },
    "entropias_bits": {claves_T[T]: entropias[T] for T in temperaturas},
    "sumas_probabilidades": {claves_T[T]: sumas[T] for T in temperaturas},
    "tabla_4_decimales": {
        "columnas": list(tabla_df.columns),
        "filas": filas,
        "markdown": tabla_markdown,
    },
    "figura": "t1_distribuciones_temperatura.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 7. Resumen breve
# ----------------------------------------------------------------------
print("\nResumen:")
for T in temperaturas:
    print(
        f"  T={T:g}: entropía = {entropias[T]:.4f} bits, "
        f"suma de probabilidades = {sumas[T]:.6f}, "
        f"token más probable = {tokens[int(np.argmax(distribuciones[T]))]}"
    )
print("Archivos generados: resultados.json, t1_distribuciones_temperatura.png")


Softmax con temperatura (distribuciones y entropías, 4 decimales):

  T     t0     t1     t2     t3     t4     t5 entropia_bits suma_probs
0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000        0.9147     1.0000
1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037        1.7998     1.0000
2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294        2.2888     1.0000

Tabla markdown:

| T | t0 | t1 | t2 | t3 | t4 | t5 | entropia_bits | suma_probs |
|---|---|---|---|---|---|---|---|---|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 | 1.0000 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 | 1.0000 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 | 1.0000 |

Resumen:
  T=0.5: entropía = 0.9147 bits, suma de probabilidades = 1.000000, token más probable = t0
  T=1: entropía = 1.7998 bits, suma de probabilidades = 1.000000, token más probable = t0
  T=2: entropía = 2.2888 bits, suma de probabilidades = 1.000000, token más probable = t0
Archi

## Parte 2 — Muestreo de núcleo (top-p)

Partiendo de la distribución de T = 1 de la Parte 1, se ordenan los tokens por probabilidad descendente y se acumula:

| rank | token | prob | prob acumulada | en núcleo |
|---|---|---|---|---|
| 1 | t0 | 0.5516 | 0.5516 | sí |
| 2 | t1 | 0.2029 | 0.7546 | sí |
| 3 | t2 | 0.1231 | 0.8776 | sí |
| 4 | t3 | 0.0912 | 0.9688 | sí |
| 5 | t4 | 0.0275 | 0.9963 | no |
| 6 | t5 | 0.0037 | 1.0000 | no |

La probabilidad acumulada con los tres primeros tokens (t0, t1, t2) es 0.8776 < 0.9; al añadir t3 la acumulada alcanza 0.9688 ≥ 0.9. Como el enunciado pide conservar el conjunto mínimo cuya probabilidad acumulada alcanza p = 0.9, el corte es inclusivo: el núcleo queda formado por {t0, t1, t2, t3} (k = 4 tokens) y quedan excluidos t4 y t5. Renormalizando entre los supervivientes (dividiendo cada probabilidad entre la acumulada del núcleo, 0.9688):

| token | probabilidad renormalizada |
|---|---|
| t0 | 0.5694 |
| t1 | 0.2095 |
| t2 | 0.1270 |
| t3 | 0.0941 |

La suma de las probabilidades renormalizadas es 1.0000. Nótese que la renormalización eleva ligeramente todas las probabilidades del núcleo respecto de la distribución original de T = 1 (por ejemplo, t0 pasa de 0.5516 a 0.5694), porque la masa excluida de t4 y t5 se reparte proporcionalmente entre los supervivientes. La figura de la celda muestra el núcleo y el corte sobre la distribución ordenada.

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==================================================================
# T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución
#      de T = 1 obtenida en T1 (Parte 1).
# ==================================================================

P = 0.9
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
FIGURA = "t2_topp_nucleo_p09.png"


def softmax_con_temperatura(logits, T):
    """Softmax con temperatura, numéricamente estable (resta del máximo)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


# ------------------------------------------------------------------
# 1) Distribución de T = 1: se toma de T1 (entrada/T1/resultados.json)
#    si está disponible; en su defecto se recalcula de forma idéntica.
# ------------------------------------------------------------------
p_t1, fuente = None, None
ruta_t1 = Path("entrada") / "T1" / "resultados.json"
if ruta_t1.exists():
    try:
        with open(ruta_t1, "r", encoding="utf-8") as f:
            t1 = json.load(f)
        p_t1 = np.asarray(t1["distribuciones"]["T=1"], dtype=float)
        fuente = str(ruta_t1)
    except Exception:
        p_t1 = None
if p_t1 is None:
    p_t1 = softmax_con_temperatura(LOGITS, 1.0)
    fuente = "recalculada con softmax numéricamente estable (T = 1)"

# ------------------------------------------------------------------
# 2) Ordenar tokens por probabilidad descendente y acumular.
# ------------------------------------------------------------------
orden = np.argsort(-p_t1, kind="stable")
probs_ord = p_t1[orden]
tokens_ord = [TOKENS[i] for i in orden]
acum = np.cumsum(probs_ord)

# ------------------------------------------------------------------
# 3) Conjunto MÍNIMO cuya probabilidad acumulada ALCANZA p = 0.9.
#    Interpretación inclusiva (la que pide el enunciado): se incluye
#    el token en cuyo paso la acumulada cruza/alcanza el umbral.
# ------------------------------------------------------------------
k = int(np.searchsorted(acum, P, side="left")) + 1
k = max(1, min(k, len(acum)))
idx_nucleo = orden[:k]
idx_fuera = orden[k:]
acum_alcanzada = float(acum[k - 1])
acum_previa = float(acum[k - 2]) if k >= 2 else 0.0

# ------------------------------------------------------------------
# 4) Renormalizar sobre los tokens supervivientes.
# ------------------------------------------------------------------
p_nucleo = p_t1[idx_nucleo]
p_renorm = p_nucleo / p_nucleo.sum()
suma_renorm = float(p_renorm.sum())

tokens_nucleo = [TOKENS[i] for i in idx_nucleo]
tokens_fuera = [TOKENS[i] for i in idx_fuera]

# ------------------- tabla ordenada (4 decimales) -----------------
en_nucleo_set = set(idx_nucleo.tolist())
filas, md_rows = [], []
for r, (tok, pv, av) in enumerate(zip(tokens_ord, probs_ord, acum), start=1):
    dentro = TOKENS.index(tok) in en_nucleo_set
    filas.append([r, tok, f"{pv:.4f}", f"{av:.4f}", "sí" if dentro else "no"])
    md_rows.append(f"| {r} | {tok} | {pv:.4f} | {av:.4f} | {'sí' if dentro else 'no'} |")
markdown_tabla = ("| rank | token | prob | prob_acumulada | en_nucleo |\n"
                  "|---|---|---|---|---|\n" + "\n".join(md_rows))

# ------------------- justificación del corte ----------------------
if k >= 2:
    justificacion = (
        f"Con los tokens ordenados de mayor a menor probabilidad "
        f"({' > '.join(tokens_ord)}), la probabilidad acumulada con los primeros "
        f"{k - 1} tokens ({', '.join(tokens_ord[:k - 1])}) es {acum_previa:.4f} < {P}, "
        f"y al añadir {tokens_ord[k - 1]} alcanza {acum_alcanzada:.4f} >= {P}. "
        f"Por tanto, el conjunto mínimo cuya probabilidad acumulada alcanza p = {P} "
        f"es {{ {', '.join(tokens_nucleo)} }} (k = {k} tokens) y el corte se justifica "
        f"con la probabilidad acumulada alcanzada = {acum_alcanzada:.4f}."
    )
else:
    justificacion = (
        f"El token más probable {tokens_ord[0]} ya alcanza una acumulada "
        f"{acum_alcanzada:.4f} >= {P}; el núcleo es {{{tokens_ord[0]}}}."
    )

# ------------------------------------------------------------------
# Resultados (contrato resultados.json)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T2_topp_nucleo_p09",
    "p": P,
    "temperatura_base": 1.0,
    "fuente_distribucion_T1": fuente,
    "tokens": TOKENS,
    "logits": LOGITS.tolist(),
    "distribucion_T1": {t: float(v) for t, v in zip(TOKENS, p_t1)},
    "interpretacion_corte": ("El enunciado pide conservar el conjunto mínimo cuya "
                             "probabilidad acumulada ALCANZA p = 0.9, por lo que se "
                             "incluye el token en cuyo paso se cruza el umbral "
                             "(corte inclusivo)."),
    "orden_descendente": tokens_ord,
    "probs_ordenadas": probs_ord.tolist(),
    "prob_acumulada_ordenada": acum.tolist(),
    "tabla_ordenada_4_decimales": {
        "columnas": ["rank", "token", "prob", "prob_acumulada", "en_nucleo"],
        "filas": filas,
    },
    "markdown_tabla_ordenada": markdown_tabla,
    "tokens_sobrevivientes": tokens_nucleo,
    "tokens_excluidos": tokens_fuera,
    "num_tokens_nucleo": k,
    "prob_acumulada_alcanzada": acum_alcanzada,
    "prob_acumulada_alcanzada_4_decimales": f"{acum_alcanzada:.4f}",
    "prob_acumulada_antes_del_ultimo_nucleo": acum_previa,
    "justificacion_corte": justificacion,
    "distribucion_renormalizada": {t: float(v) for t, v in zip(tokens_nucleo, p_renorm)},
    "distribucion_renormalizada_orden_tokens": tokens_nucleo,
    "distribucion_renormalizada_valores": p_renorm.tolist(),
    "distribucion_renormalizada_4_decimales": {
        t: f"{v:.4f}" for t, v in zip(tokens_nucleo, p_renorm)
    },
    "suma_probs_renormalizadas": suma_renorm,
    "suma_probs_renormalizadas_4_decimales": f"{suma_renorm:.4f}",
    "verificaciones": {
        "suma_renormalizada_es_1": bool(abs(suma_renorm - 1.0) < 1e-9),
        "acumulada_alcanza_p": bool(acum_alcanzada >= P),
        "acumulada_previa_menor_que_p": bool(acum_previa < P),
        "nucleo_es_conjunto_minimo": bool(acum_previa < P <= acum_alcanzada),
    },
    "figura": FIGURA,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# Figura: distribución T=1, acumulación con corte y renormalizada
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))

ax = axes[0]
mascara = np.zeros(len(TOKENS), dtype=bool)
mascara[idx_nucleo] = True
colores = ["#2a9d8f" if m else "#d64545" for m in mascara]
ax.bar(TOKENS, p_t1, color=colores, edgecolor="black", linewidth=0.6)
for i, v in enumerate(p_t1):
    ax.text(i, v + 0.012, f"{v:.4f}", ha="center", fontsize=8)
ax.set_ylim(0, float(p_t1.max()) * 1.22)
ax.set_ylabel("p_i  (T = 1)")
ax.set_title("Distribución T=1\n(verde = núcleo, rojo = excluido)", fontsize=10)

ax = axes[1]
ranks = np.arange(1, len(TOKENS) + 1)
ax.step(ranks, acum, where="post", marker="o", color="#264653", label="prob. acumulada")
ax.axhline(P, color="#e9c46a", linestyle="--", linewidth=1.5, label=f"p = {P}")
ax.axvline(k, color="#d64545", linestyle=":", linewidth=1.5, label=f"corte: k = {k}")
ax.scatter([k], [acum_alcanzada], color="#d64545", zorder=5)
ax.annotate(f"{acum_alcanzada:.4f}", (k, acum_alcanzada),
            textcoords="offset points", xytext=(10, -12),
            fontsize=9, color="#d64545")
ax.set_xticks(ranks)
ax.set_xticklabels(tokens_ord)
ax.set_ylim(0, 1.06)
ax.set_xlabel("tokens ordenados (prob. descendente)")
ax.set_ylabel("probabilidad acumulada")
ax.set_title("Acumulación y corte del núcleo", fontsize=10)
ax.legend(fontsize=8, loc="lower right")

ax = axes[2]
ax.bar(tokens_nucleo, p_renorm, color="#2a9d8f", edgecolor="black", linewidth=0.6)
for i, v in enumerate(p_renorm):
    ax.text(i, v + 0.012, f"{v:.4f}", ha="center", fontsize=8)
ax.set_ylim(0, float(p_renorm.max()) * 1.22)
ax.set_ylabel("p_i renormalizada")
ax.set_title(f"Distribución renormalizada\n(suma = {suma_renorm:.4f})", fontsize=10)

fig.suptitle(f"Top-p (nucleus sampling) con p = {P} sobre la distribución de T = 1",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(FIGURA, dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print("=" * 66)
print("T2 — Top-p (nucleus sampling) con p = 0.9 sobre la distribución T = 1")
print("=" * 66)
print(f"Fuente distribución T=1     : {fuente}")
print("Orden descendente           : "
      + " > ".join(f"{t}({v:.4f})" for t, v in zip(tokens_ord, probs_ord)))
print("Prob. acumulada por rango   : "
      + " | ".join(f"{a:.4f}" for a in acum))
print(f"Tokens supervivientes (k={k})  : {tokens_nucleo}")
print(f"Tokens excluidos            : {tokens_fuera}")
print(f"Prob. acumulada alcanzada   : {acum_alcanzada:.6f}  (>= p = {P}; "
      f"previa = {acum_previa:.6f} < p)")
print("Distribución renormalizada  : "
      + ", ".join(f"{t}={v:.6f}" for t, v in zip(tokens_nucleo, p_renorm)))
print(f"Suma renormalizada          : {suma_renorm:.12f}")
print(f"Figura                      : {FIGURA}")
print("Resultados                  : resultados.json")


T2 — Top-p (nucleus sampling) con p = 0.9 sobre la distribución T = 1
Fuente distribución T=1     : entrada/T1/resultados.json
Orden descendente           : t0(0.5516) > t1(0.2029) > t2(0.1231) > t3(0.0912) > t4(0.0275) > t5(0.0037)
Prob. acumulada por rango   : 0.5516 | 0.7546 | 0.8776 | 0.9688 | 0.9963 | 1.0000
Tokens supervivientes (k=4)  : ['t0', 't1', 't2', 't3']
Tokens excluidos            : ['t4', 't5']
Prob. acumulada alcanzada   : 0.968820  (>= p = 0.9; previa = 0.877637 < p)
Distribución renormalizada  : t0=0.569376, t1=0.209462, t2=0.127045, t3=0.094117
Suma renormalizada          : 1.000000000000
Figura                      : t2_topp_nucleo_p09.png
Resultados                  : resultados.json


## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0), mediante rng.choice sobre los cuatro tokens del núcleo con sus probabilidades renormalizadas. Las frecuencias observadas frente a las probabilidades teóricas son:

| token | p teórica | conteo observado | frecuencia observada |
|---|---|---|---|
| t0 | 0.5694 | 5682 | 0.5682 |
| t1 | 0.2095 | 2093 | 0.2093 |
| t2 | 0.1270 | 1296 | 0.1296 |
| t3 | 0.0941 | 929 | 0.0929 |

Las frecuencias observadas suman 1.0000 y siguen de cerca a las teóricas: en t0, t1 y t3 la frecuencia observada coincide con la teórica hasta la tercera o cuarta cifra (0.5682 frente a 0.5694, 0.2093 frente a 0.2095, 0.0929 frente a 0.0941), y en t2 se observa 0.1296 frente a 0.1270 teórico. El gráfico de barras generado en la celda compara ambas series y muestra que las barras de frecuencias observadas son visualmente indistinguibles de las teóricas. La divergencia KL de la distribución empírica respecto de la teórica, KL(q‖p) = Σ_i q_i · log2(q_i / p_i) con q la empírica y p la teórica, es de 5.0069e-05 bits (0.0001 a cuatro decimales). Este valor, prácticamente nulo, confirma que con 10 000 muestras la distribución empírica reproduce a la teórica y que las diferencias observadas son solo ruido de muestreo.

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
Subtarea T3 — Comprobación empírica del muestreo del núcleo (top-p, p = 0.9, T = 1)

1) Carga la distribución renormalizada de T2 (entrada/T2/resultados.json).
2) Toma 10 000 muestras con numpy.random.default_rng(0).
3) Grafica frecuencias observadas vs. probabilidades teóricas (diagrama de barras).
4) Calcula la divergencia KL en bits de la distribución empírica respecto de la teórica.
5) Escribe resultados.json (contrato de la subtarea).
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# 1) Distribución renormalizada de T2 (distribución teórica de referencia)
# ----------------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    tokens_nucleo = list(t2["distribucion_renormalizada_orden_tokens"])
    p_teorica = np.array(t2["distribucion_renormalizada_valores"], dtype=float)
    fuente = "entrada/T2/resultados.json"
else:
    # Respaldo: recalcular desde el enunciado (softmax T=1 + top-p p=0.9, corte inclusivo)
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p_full = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_full)
    acum = np.cumsum(p_full[orden])
    k = int(np.searchsorted(acum, 0.9) + 1)  # conjunto mínimo que ALCANZA p=0.9
    nucleo_idx = np.sort(orden[:k])
    tokens_nucleo = [tokens[i] for i in nucleo_idx]
    p_teorica = p_full[nucleo_idx] / p_full[nucleo_idx].sum()
    fuente = "recalculada desde logits (softmax T=1, top-p p=0.9 inclusivo)"

p_teorica = p_teorica / p_teorica.sum()  # seguridad numérica

# ----------------------------------------------------------------------------
# 2) Muestreo: 10 000 muestras con numpy.random.default_rng(0)
# ----------------------------------------------------------------------------
N_MUESTRAS = 10_000
SEMILLA = 0
rng = np.random.default_rng(SEMILLA)
indices = rng.choice(len(tokens_nucleo), size=N_MUESTRAS, p=p_teorica)

# ----------------------------------------------------------------------------
# 3) Distribución empírica (frecuencias observadas)
# ----------------------------------------------------------------------------
conteos = np.bincount(indices, minlength=len(tokens_nucleo)).astype(int)
frec_obs = conteos / N_MUESTRAS

# ----------------------------------------------------------------------------
# 4) Divergencia KL en bits: KL(empírica || teórica) = Σ q_i log2(q_i / p_i)
# ----------------------------------------------------------------------------
mask = frec_obs > 0
kl_bits = float(np.sum(frec_obs[mask] * np.log2(frec_obs[mask] / p_teorica[mask])))

# ----------------------------------------------------------------------------
# 5) Diagrama de barras: frecuencias observadas vs. probabilidades teóricas
# ----------------------------------------------------------------------------
nombre_figura = "t3_frecuencias_vs_teoricas.png"
x = np.arange(len(tokens_nucleo))
ancho = 0.38

fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, frec_obs, width=ancho,
            label=f"Frecuencia observada ({N_MUESTRAS:,} muestras)".replace(",", " "),
            color="#4C72B0", edgecolor="black")
b2 = ax.bar(x + ancho / 2, p_teorica, width=ancho,
            label="Probabilidad teórica (top-p renormalizada)",
            color="#DD8452", edgecolor="black")
for rect in list(b1) + list(b2):
    h = rect.get_height()
    ax.annotate(f"{h:.4f}", (rect.get_x() + rect.get_width() / 2, h),
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(tokens_nucleo)
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_xlabel("Token del núcleo")
ax.set_title(
    "Parte 3 — Comprobación empírica del muestreo top-p (T=1, p=0.9)\n"
    f"n = 10 000, rng = default_rng(0) · KL(empírica ‖ teórica) = {kl_bits:.6f} bits"
)
ax.set_ylim(0, max(frec_obs.max(), p_teorica.max()) * 1.18)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(nombre_figura, dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------------
# 6) Contrato: resultados.json
# ----------------------------------------------------------------------------
resultados = {
    "subtarea": "T3_comprobacion_empirica_kl_bits",
    "n_muestras": N_MUESTRAS,
    "semilla": SEMILLA,
    "generador": "numpy.random.default_rng(0)",
    "metodo_muestreo": "rng.choice(k, size=10000, p=distribucion_renormalizada)",
    "fuente_distribucion_teorica": fuente,
    "tokens_nucleo": tokens_nucleo,
    "p_teorica_valores": p_teorica.tolist(),
    "p_teorica": {tok: float(p) for tok, p in zip(tokens_nucleo, p_teorica)},
    "p_teorica_4_decimales": {tok: f"{p:.4f}" for tok, p in zip(tokens_nucleo, p_teorica)},
    "conteos_observados_valores": conteos.tolist(),
    "conteos_observados": {tok: int(c) for tok, c in zip(tokens_nucleo, conteos)},
    "frecuencias_observadas_valores": frec_obs.tolist(),
    "frecuencias_observadas": {tok: float(fr) for tok, fr in zip(tokens_nucleo, frec_obs)},
    "frecuencias_observadas_4_decimales": {tok: f"{fr:.4f}" for tok, fr in zip(tokens_nucleo, frec_obs)},
    "suma_frecuencias_observadas": float(frec_obs.sum()),
    "formula_kl": "KL(q||p) = sum_i q_i * log2(q_i / p_i), q = empírica, p = teórica (bits)",
    "kl_bits_empirica_respecto_teorica": kl_bits,
    "kl_bits": kl_bits,
    "kl_bits_4_decimales": f"{kl_bits:.4f}",
    "figura": nombre_figura,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------------
print("=== Subtarea T3 — Comprobación empírica (top-p p=0.9, T=1) ===")
print(f"Muestras: {N_MUESTRAS} con numpy.random.default_rng({SEMILLA})")
print("Token | conteo | frec. observada | prob. teórica")
for tok, c, fo, pt in zip(tokens_nucleo, conteos, frec_obs, p_teorica):
    print(f"  {tok}  | {c:6d} |      {fo:.4f}     |    {pt:.4f}")
print(f"Suma frecuencias observadas: {frec_obs.sum():.6f}")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print(f"Figura guardada: {nombre_figura}")
print("Contrato escrito: resultados.json")


=== Subtarea T3 — Comprobación empírica (top-p p=0.9, T=1) ===
Muestras: 10000 con numpy.random.default_rng(0)
Token | conteo | frec. observada | prob. teórica
  t0  |   5682 |      0.5682     |    0.5694
  t1  |   2093 |      0.2093     |    0.2095
  t2  |   1296 |      0.1296     |    0.1270
  t3  |    929 |      0.0929     |    0.0941
Suma frecuencias observadas: 1.000000
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: t3_frecuencias_vs_teoricas.png
Contrato escrito: resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0⁺, el cociente z_i/T amplifica las diferencias entre logits: para todo token con z_i < z_max se tiene exp((z_i − z_max)/T) → 0, de modo que toda la masa de probabilidad se concentra en el token de mayor logit (aquí t0, con z = 2.0) y la distribución tiende a una masa puntual (delta) sobre el argmax. Las entropías de la Parte 1 muestran esa tendencia: la entropía cae de 2.2888 bits (T = 2) a 1.7998 bits (T = 1) y a 0.9147 bits (T = 0.5), y con T = 0.5 el token t0 ya concentra 0.8231 de la masa; al seguir reduciendo T la entropía continúa descendiendo hacia su mínimo posible, 0 bits.

Cuando T → ∞, en cambio, z_i/T → 0 para todo i, todos los exponenciales se igualan y p_i → 1/6: la distribución tiende a la uniforme sobre el vocabulario. Esto es coherente con la evidencia medida: la entropía crece con T (0.9147 → 1.7998 → 2.2888 bits) acercándose al máximo teórico log2(6), que es precisamente la entropía de la distribución uniforme.

Que T → 0 equivalga a la decodificación voraz (greedy) se sigue de la definición de esta: greedy selecciona determinísticamente el token de mayor logit, es decir, el argmax. En el límite T → 0 la softmax asigna probabilidad 1 a ese token y 0 al resto, de modo que cualquier muestreo de esa distribución devuelve siempre el argmax, exactamente igual que greedy. La entropía nula del límite refleja esa ausencia total de aleatoriedad: no queda incertidumbre que el muestreo pueda explotar, y por eso tanto muestrear con T → 0 como decodificar vorazmente producen la misma salida.